# Two-Wheeler Violation Detection — Complete Training Pipeline (v3)

## Architecture: Cascade of 4 Detectors

| Sub-dataset | What it detects | Classes |
|---|---|---|
| `twowheeler_train_val` | Find the two-wheeler | `twowheeler` |
| `rider_train_val` | Find the rider | `rider` |
| `helmet_no_helmet_train_val` | Classify helmet status | `WithHelmet`, `WithoutHelmet` |
| `plate_train_val` | Detect the license plate | `Plate` |

---
### 🔬 Project Novelty: CCTV Degradation Robustness Benchmark
Trains all 4 core models + re-trains an augmented rider model against simulated adverse conditions (**Night**, **Motion Blur**, **Rain**, and **JPEG Compression**) to prove superior real-world performance.

### 🚀 Colab & Kaggle Ready
- **Google Colab**: Switch runtime to GPU (**Runtime → Change runtime type → T4 GPU**).
- **Dataset**: Upload `two-wheeler-violation-and-license-detection.zip` (or attach dataset in Kaggle). The notebook automatically unpacks and locates all subsets.

## Step 1 — Install Dependencies & Configure Environment
Detects whether running in **Google Colab**, **Kaggle**, or **Local**, sets up output directories, and checks GPU availability.

In [ ]:
# Install Ultralytics
!pip install -q ultralytics

import os, sys, glob, random, yaml, shutil, zipfile
from pathlib import Path
import numpy as np
import torch, cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO, settings

# Disable wandb logging
settings.update({"wandb": False})

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Detect platform environment
IS_COLAB = "google.colab" in sys.modules or Path("/content").exists()
IS_KAGGLE = Path("/kaggle/working").exists()

if IS_COLAB:
    BASE_DIR = Path("/content")
    ENV_NAME = "Google Colab"
elif IS_KAGGLE:
    BASE_DIR = Path("/kaggle/working")
    ENV_NAME = "Kaggle"
else:
    BASE_DIR = Path(".").resolve()
    ENV_NAME = "Local Environment"

RUNS_DIR = BASE_DIR / "runs"
YAMLS_DIR = BASE_DIR / "yamls"
DEGRADED_DIR = BASE_DIR / "degraded"
EXPORT_DIR = BASE_DIR / "exported_weights"

for d in [RUNS_DIR, YAMLS_DIR, DEGRADED_DIR, EXPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print(f" Environment : {ENV_NAME}")
print(f" Working Dir : {BASE_DIR}")
print(f" Runs Dir    : {RUNS_DIR}")
print("=" * 60)

# Device check & warning
has_cuda = torch.cuda.is_available()
if has_cuda:
    DEVICE = 0
    print(f" CUDA Status : Available (GPU: {torch.cuda.get_device_name(0)})")
else:
    DEVICE = "cpu"
    print(f" CUDA Status : NOT Available (Using CPU)")
    print("\n" + "!" * 65)
    print(" ⚠️  WARNING: NO GPU DETECTED!")
    print(" Training YOLO on CPU is extremely slow (10+ hours per model).")
    if IS_COLAB:
        print(" 👉 IN GOOGLE COLAB:")
        print("    1. Click 'Runtime' in top menu -> 'Change runtime type'")
        print("    2. Under 'Hardware accelerator', select 'T4 GPU'")
        print("    3. Click 'Save' and restart/rerun this notebook.")
    print("!" * 65 + "\n")

# Hyperparameters
# 80 epochs for full research-grade convergence on GPU, 3 epochs for quick CPU testing
EPOCHS = 80 if has_cuda else 3
BATCH_SIZE = 16 if has_cuda else 4
WORKERS = 2 if has_cuda else 0

print(f"Training Config: EPOCHS={EPOCHS}, BATCH_SIZE={BATCH_SIZE}, WORKERS={WORKERS}, DEVICE={DEVICE}")

## Step 2 — Automatic Dataset Extraction & Discovery
Automatically unzips any uploaded dataset zip (e.g., `two-wheeler-violation-and-license-detection.zip`) and locates all 4 sub-datasets.

In [ ]:
# 1. Extract dataset zip if present
zip_candidates = list(BASE_DIR.glob("*.zip")) + list(Path(".").glob("*.zip"))
for z in zip_candidates:
    if any(k in z.name.lower() for k in ["two-wheeler", "violation", "dataset", "license", "helmet"]):
        extract_target = BASE_DIR / "dataset"
        print(f"Found dataset archive: {z.name}")
        # Avoid extracting twice if sub-datasets already exist
        if not (extract_target / "rider_train_val").exists() and not (extract_target / "Dataset" / "rider_train_val").exists():
            print(f"Extracting {z.name} to {extract_target} ...")
            with zipfile.ZipFile(z, 'r') as zip_ref:
                zip_ref.extractall(extract_target)
            print("Extraction complete!")
        else:
            print("Dataset folder already exists, skipping re-extraction.")
        break

# 2. Scan candidate directories for dataset.yaml
candidate_search_paths = [
    BASE_DIR,
    BASE_DIR / "dataset",
    Path("/kaggle/input"),
    Path("/content"),
    Path("../dataset"),
    Path("./dataset"),
    Path("."),
]

found_yamls = []
for sp in candidate_search_paths:
    if sp.exists():
        found_yamls.extend(sp.rglob("dataset.yaml"))

SUBSETS = {}
for y in sorted(set(found_yamls)):
    y_str = str(y).replace("\\", "/")
    # Skip YAMLs inside output or degraded folders
    if any(ig in y_str for ig in ["/runs/", "/yamls/", "/degraded/"]):
        continue
    try:
        c = yaml.safe_load(open(y, encoding="utf-8"))
        if not c or "names" not in c:
            continue
        nm = c["names"]
        nm = [nm[k] for k in sorted(nm)] if isinstance(nm, dict) else list(nm)
        key = y.parent.name
        if any(sub in key for sub in ["rider", "helmet", "plate", "twowheeler", "two_wheeler"]):
            SUBSETS[key] = (y, nm)
            print(f"[+] Sub-dataset found: {key:30s} | classes={nm}")
    except Exception as e:
        pass

print(f"\nTotal discovered sub-datasets: {len(SUBSETS)}")
if len(SUBSETS) < 4:
    print("⚠️ Notice: Expected 4 sub-datasets. Check that your zip file or dataset directory is in the workspace.")

## Step 3 — Build Verified YAML Configurations
Creates verified YAML configs with absolute image paths and confirms that training/validation images and labels exist.

In [ ]:
def prepare(key):
    """
    Builds a standalone, verified YAML config with absolute paths.
    Works cross-platform across Colab, Kaggle, Windows, and Linux.
    """
    if key not in SUBSETS:
        raise KeyError(f"Sub-dataset '{key}' not found in SUBSETS. Available: {list(SUBSETS.keys())}")
    
    ypath, names = SUBSETS[key]
    root = ypath.parent
    cfg = yaml.safe_load(open(ypath, encoding="utf-8"))
    
    # Resolve train path
    train_rel = cfg.get("train", "train/images")
    train_path = root / train_rel if not Path(train_rel).is_absolute() else Path(train_rel)
    if not train_path.exists():
        train_path = root / "train" / "images"
        
    # Resolve val path
    val_rel = cfg.get("val", "val/images")
    val_path = root / val_rel if not Path(val_rel).is_absolute() else Path(val_rel)
    if not val_path.exists():
        val_path = root / "val" / "images"
        
    out = {
        "path" : "",
        "train": str(train_path).replace("\\", "/"),
        "val"  : str(val_path).replace("\\", "/"),
        "nc"   : len(names),
        "names": names,
    }
    
    if cfg.get("test"):
        test_path = root / cfg["test"] if not Path(cfg["test"]).is_absolute() else Path(cfg["test"])
        if test_path.exists():
            out["test"] = str(test_path).replace("\\", "/")
            
    for s in ("train", "val"):
        p = Path(out[s])
        assert p.exists(), f"[{key}] {s} images directory missing: {p}"
        n_imgs = sum(f.suffix.lower() in IMG_EXT for f in p.iterdir())
        lbl_dir = p.parent / "labels"
        print(f"[{key}] {s:5s}: {n_imgs:>4d} images | labels_dir={lbl_dir.exists()} ({lbl_dir.name})")
        
    dst = YAMLS_DIR / f"{key}.yaml"
    with open(dst, "w", encoding="utf-8") as f:
        yaml.dump(out, f, sort_keys=False)
    print(f"  -> Wrote YAML: {dst}\n")
    return str(dst)

# Verify and write YAML configs for all discovered sub-datasets
for key in SUBSETS:
    prepare(key)

## Step 4 — Epoch Loss & Metric Monitoring Callbacks

In [ ]:
def on_train_epoch_end(trainer):
    items = trainer.label_loss_items(trainer.tloss, prefix="train")
    msg = " | ".join(f"{k}: {v:.4f}" for k, v in items.items())
    print(f"[Epoch {trainer.epoch + 1:>3}/{trainer.epochs}] {msg}")

def on_fit_epoch_end(trainer):
    val_losses = {k: v for k, v in trainer.metrics.items() if "loss" in k}
    msg = " | ".join(f"{k}: {v:.4f}" for k, v in val_losses.items())
    map50 = trainer.metrics.get("metrics/mAP50(B)", 0)
    print(f"{'':>30}VAL -> {msg}  |  mAP50: {map50:.4f}")

def attach(model):
    model.add_callback("on_train_epoch_end", on_train_epoch_end)
    model.add_callback("on_fit_epoch_end",   on_fit_epoch_end)
    return model

print("Callbacks ready.")

## Step 5 — Training Function & Weights Registry (`BESTS`)
Maintains a centralized registry `BESTS` storing weights for all trained detectors.

In [ ]:
BESTS = {}

# Automatically scan for existing weights from previous runs
for key in ["rider_train_val", "helmet_no_helmet_train_val", "plate_train_val", "twowheeler_train_val", "rider_augmented"]:
    candidates = [
        RUNS_DIR / key / "weights" / "best.pt",
        RUNS_DIR / "full" / "weights" / "best.pt" if key == "rider_train_val" else None,
        BASE_DIR / "weights" / f"{key.split('_')[0]}_best.pt",
    ]
    for cp in candidates:
        if cp and cp.exists():
            BESTS[key] = str(cp)
            print(f"Loaded existing weights for [{key}]: {cp}")
            break

def train_subset(key, epochs=EPOCHS, weights="yolov8s.pt", imgsz=640, batch=BATCH_SIZE, **kw):
    """Train one sub-dataset detector and record the best-weight path in BESTS."""
    data = prepare(key)
    m = attach(YOLO(weights))
    _ = m.train(
        data         = data,
        epochs       = epochs,
        imgsz        = imgsz,
        batch        = batch,
        workers      = WORKERS,
        device       = DEVICE,
        patience     = 20,
        close_mosaic = 10,
        project      = str(RUNS_DIR),
        name         = key,
        exist_ok     = True,
        plots        = True,
        **kw,
    )
    best_path = str(m.trainer.best)
    BESTS[key] = best_path
    print(f"\n[{key}] Training complete! Best weights saved: {best_path}")
    return best_path

print("Training function defined.")

## Step 6.1 — Train Baseline Rider Detector (`rider_train_val`)
Detects motorcycle riders. If pre-trained weights exist in `runs/`, it skips re-training.

In [ ]:
if "rider_train_val" not in BESTS:
    print("Starting baseline rider detector training...")
    train_subset("rider_train_val", epochs=EPOCHS, batch=BATCH_SIZE)
else:
    print(f"Rider weights already present: {BESTS['rider_train_val']}")

## Step 6.2 — Train Helmet / No-Helmet Classifier (`helmet_no_helmet_train_val`)
**Primary violation detector** — classifies whether riders are wearing helmets (`WithHelmet`) or not (`WithoutHelmet`).

In [ ]:
train_subset(
    "helmet_no_helmet_train_val",
    epochs = EPOCHS,
    batch  = BATCH_SIZE,
)

## Step 6.3 — Train License Plate Detector (`plate_train_val`)
License plates are small objects. Trained with higher resolution (`imgsz=960`) for precision.

In [ ]:
# For small license plates: higher resolution (960px) with halved batch size
plate_batch = max(2, BATCH_SIZE // 2) if DEVICE != "cpu" else 2
train_subset(
    "plate_train_val",
    epochs = EPOCHS,
    imgsz  = 960,
    batch  = plate_batch,
)

## Step 6.4 — Train Two-Wheeler Detector (`twowheeler_train_val`)
Initial stage detector — identifies two-wheelers in the scene.

In [ ]:
train_subset(
    "twowheeler_train_val",
    epochs = EPOCHS,
    batch  = BATCH_SIZE,
)

## Step 7 — Validate All Models
Evaluates Precision, Recall, mAP@50, and mAP@50-95 for each trained model.

In [ ]:
for key, w in list(BESTS.items()):
    if key == "rider_augmented":
        continue
    yaml_path = YAMLS_DIR / f"{key}.yaml"
    if not yaml_path.exists():
        yaml_path = Path(prepare(key))
    imgsz = 960 if key.startswith("plate") else 640
    mdl = YOLO(w)
    m = mdl.val(data=str(yaml_path), imgsz=imgsz, device=DEVICE, verbose=False)
    print(f"\n{'='*55}")
    print(f"  Model: {key}")
    print(f"  Precision : {m.box.mp:.4f}")
    print(f"  Recall    : {m.box.mr:.4f}")
    print(f"  mAP50     : {m.box.map50:.4f}")
    print(f"  mAP50-95  : {m.box.map:.4f}")
    print(f"  Per-class (mAP50-95):")
    for i, c in enumerate(m.box.ap_class_index):
        print(f"    {mdl.names[c]:20s}  {m.box.maps[i]:.4f}")

## Step 8 — Visual Inference on Validation Images
Visual sample predictions for each model.

In [ ]:
from IPython.display import display as ipy_display

for key, w in list(BESTS.items()):
    if key == "rider_augmented":
        continue
    yaml_path = YAMLS_DIR / f"{key}.yaml"
    if not yaml_path.exists():
        prepare(key)
    cfg = yaml.safe_load(open(yaml_path, encoding="utf-8"))
    img_dir = Path(cfg.get("test", cfg["val"]))
    imgs = [p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXT]
    if not imgs:
        continue
    samples = random.sample(imgs, min(3, len(imgs)))
    imgsz = 960 if key.startswith("plate") else 640
    results = YOLO(w).predict(
        source  = [str(p) for p in samples],
        conf    = 0.25,
        imgsz   = imgsz,
        device  = DEVICE,
        verbose = False,
    )
    fig, axes = plt.subplots(1, len(results), figsize=(7 * len(results), 5))
    if len(results) == 1:
        axes = [axes]
    for ax, r in zip(axes, results):
        ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
        ax.set_title(Path(r.path).name, fontsize=8)
        ax.axis("off")
    fig.suptitle(key, fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.show()

## Step 9 — Training Curves & Confusion Matrices

In [ ]:
from IPython.display import Image as IPyImage, display as ipy_display

for key, w in list(BESTS.items()):
    run_dir = Path(w).parent.parent
    print(f"\n--- Model: {key} ---")
    for fname in ["results.png", "confusion_matrix.png"]:
        fpath = run_dir / fname
        if fpath.exists():
            ipy_display(IPyImage(str(fpath)))
        else:
            print(f"  {fname} not found in {run_dir}")

---
# 🔬 NOVELTY SECTION: Degradation-Aware Training & Benchmark

This section benchmarks our system against real-world CCTV degradation:
1. Train an **augmented rider detector** using CCTV-realistic augmentations (heavy HSV brightness jitter, rotation, scale, mixup, random erasing).
2. Generate synthetic degraded validation sets (**Night**, **Motion Blur**, **Rain**, and **JPEG Compression**).
3. Benchmark **Baseline vs Augmented** models side-by-side to demonstrate resilience.

## Step 10 — Train CCTV-Augmented Rider Model

In [ ]:
# ===== NOVELTY: Degradation-Aware Training =====
# Train rider model with CCTV-realistic augmentations

model_aug = attach(YOLO("yolov8s.pt"))
_ = model_aug.train(
    data         = prepare("rider_train_val"),
    epochs       = EPOCHS,
    imgsz        = 640,
    batch        = BATCH_SIZE,
    workers      = WORKERS,
    device       = DEVICE,
    patience     = 20,
    close_mosaic = 10,
    # CCTV degradation augmentations
    hsv_h        = 0.015,
    hsv_s        = 0.7,
    hsv_v        = 0.6,       # Strong brightness jitter (simulates dusk/night)
    degrees      = 5,         # Slight camera tilt
    scale        = 0.7,       # Near vs far vehicles
    mixup        = 0.1,       # Overlapping traffic simulation
    erasing      = 0.4,       # Random occlusion by other vehicles/poles
    fliplr       = 0.5,
    project      = str(RUNS_DIR),
    name         = "rider_augmented",
    exist_ok     = True,
    plots        = True,
)

AUG_WEIGHTS = str(model_aug.trainer.best)
BESTS["rider_augmented"] = AUG_WEIGHTS
print(f"\nAugmented rider weights saved: {AUG_WEIGHTS}")

## Step 11 — Build Degraded Validation Sets (Night, Blur, Rain, Compression)
Applies synthetic real-world distortions to validation images.

In [ ]:
# ===== DEGRADATION SIMULATION FUNCTIONS =====

def simulate_night(im):
    """Gamma darken + sensor noise to simulate nighttime CCTV"""
    dark = (255 * ((im / 255.0) ** 2.5) * 0.6).astype(np.uint8)
    noisy = np.clip(dark + np.random.normal(0, 6, dark.shape), 0, 255)
    return noisy.astype(np.uint8)

def simulate_blur(im):
    """Horizontal motion blur simulating moving vehicle / camera vibration"""
    kernel = np.zeros((15, 15))
    kernel[7, :] = 1.0 / 15
    return cv2.filter2D(im, -1, kernel)

def simulate_rain(im):
    """Add rain streaks simulating wet weather conditions"""
    rain = np.zeros_like(im)
    for _ in range(300):
        x = np.random.randint(0, im.shape[1])
        y = np.random.randint(0, im.shape[0])
        length = np.random.randint(10, 30)
        cv2.line(rain, (x, y), (x + 1, y + length), (200, 200, 200), 1)
    return cv2.addWeighted(im, 0.85, rain, 0.15, 0)

def simulate_compression(im):
    """Low-bitrate JPEG compression artifacts common in low-cost CCTV streams"""
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), 15]
    _, buf = cv2.imencode('.jpg', im, encode_param)
    return cv2.imdecode(buf, cv2.IMREAD_COLOR)

def build_degraded_val(key, condition_name, transform_fn):
    """Create a degraded copy of the val set and return its yaml path"""
    yaml_src = YAMLS_DIR / f"{key}.yaml"
    if not yaml_src.exists():
        prepare(key)
    base = yaml.safe_load(open(yaml_src, encoding="utf-8"))
    src_img = Path(base["val"])
    src_lbl = src_img.parent / "labels"
    
    out_img = DEGRADED_DIR / key / condition_name / "images"
    out_lbl = out_img.parent / "labels"
    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)
    
    for p in src_img.iterdir():
        if p.suffix.lower() in IMG_EXT:
            dst_im_path = out_img / p.name
            if not dst_im_path.exists():
                img = cv2.imread(str(p))
                if img is not None:
                    degraded = transform_fn(img)
                    cv2.imwrite(str(dst_im_path), degraded)
            lbl = src_lbl / f"{p.stem}.txt"
            if lbl.exists() and not (out_lbl / lbl.name).exists():
                shutil.copy(lbl, out_lbl / lbl.name)
    
    base["val"] = str(out_img).replace("\\", "/")
    dst = YAMLS_DIR / f"{key}_{condition_name}.yaml"
    with open(dst, "w", encoding="utf-8") as f:
        yaml.dump(base, f, sort_keys=False)
    return str(dst)

print("Degradation functions ready.")

# Visual preview of degradation effects
rider_cfg = yaml.safe_load(open(YAMLS_DIR / "rider_train_val.yaml", encoding="utf-8"))
sample_img_paths = [p for p in Path(rider_cfg["val"]).iterdir() if p.suffix.lower() in IMG_EXT]
if sample_img_paths:
    sample_img = cv2.imread(str(sample_img_paths[0]))
    fig, axes = plt.subplots(1, 5, figsize=(25, 5))
    conditions = [
        ("Original", lambda x: x),
        ("Night", simulate_night),
        ("Motion Blur", simulate_blur),
        ("Rain", simulate_rain),
        ("JPEG Compression", simulate_compression),
    ]
    for ax, (name, fn) in zip(axes, conditions):
        ax.imshow(cv2.cvtColor(fn(sample_img.copy()), cv2.COLOR_BGR2RGB))
        ax.set_title(name, fontsize=12, fontweight="bold")
        ax.axis("off")
    plt.suptitle("Simulated CCTV Degradation Conditions", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

## Step 12 — Run Degradation Benchmark: Baseline vs Augmented
Evaluates both models across all 5 conditions to generate empirical research results.

In [ ]:
# ===== DEGRADATION BENCHMARK =====
key = "rider_train_val"
conditions = {
    "clean":       lambda x: x,
    "night":       simulate_night,
    "blur":        simulate_blur,
    "rain":        simulate_rain,
    "compressed":  simulate_compression,
}

# Safe resolution of baseline rider weights
baseline_weights = BESTS.get("rider_train_val")
if not baseline_weights or not Path(baseline_weights).exists():
    for cand in [RUNS_DIR / "rider_train_val" / "weights" / "best.pt", RUNS_DIR / "full" / "weights" / "best.pt"]:
        if cand.exists():
            baseline_weights = str(cand)
            BESTS["rider_train_val"] = baseline_weights
            break

# Safe resolution of augmented rider weights
augmented_weights = globals().get("AUG_WEIGHTS") or BESTS.get("rider_augmented")
if not augmented_weights or not Path(augmented_weights).exists():
    cand = RUNS_DIR / "rider_augmented" / "weights" / "best.pt"
    if cand.exists():
        augmented_weights = str(cand)
        BESTS["rider_augmented"] = augmented_weights

print(f"Baseline weights  : {baseline_weights}")
print(f"Augmented weights : {augmented_weights}")

assert baseline_weights and Path(baseline_weights).exists(), "Baseline rider weights not found! Please run Step 6.1 first."
assert augmented_weights and Path(augmented_weights).exists(), "Augmented rider weights not found! Please run Step 10 first."

models = {
    "Baseline":  baseline_weights,
    "Augmented": augmented_weights,
}

results_table = []
print(f"\n{'Model':<12} {'Condition':<14} {'mAP50':>8} {'Recall':>8} {'Precision':>10}")
print("-" * 56)

for tag, weights in models.items():
    for cond, fn in conditions.items():
        yaml_path = build_degraded_val(key, cond, fn)
        r = YOLO(weights).val(data=yaml_path, device=DEVICE, verbose=False)
        print(f"{tag:<12} {cond:<14} {r.box.map50:>8.3f} {r.box.mr:>8.3f} {r.box.mp:>10.3f}")
        results_table.append({
            "Model": tag,
            "Condition": cond,
            "mAP50": round(float(r.box.map50), 3),
            "Recall": round(float(r.box.mr), 3),
            "Precision": round(float(r.box.mp), 3),
        })

print("\n" + "=" * 56)
print("RESEARCH SUMMARY TABLE (READY FOR PAPER / PRESENTATION)")
print("=" * 56)
import pandas as pd
df_results = pd.DataFrame(results_table)
display(df_results)

## Step 13 — Visualize Baseline vs Augmented on Degraded Images
Side-by-side detection comparisons in challenging conditions (Night & Blur).

In [ ]:
rider_cfg = yaml.safe_load(open(YAMLS_DIR / f"{key}.yaml", encoding="utf-8"))
val_imgs = [p for p in Path(rider_cfg["val"]).iterdir() if p.suffix.lower() in IMG_EXT]
samples = random.sample(val_imgs, min(3, len(val_imgs)))

baseline_model = YOLO(baseline_weights)
aug_model = YOLO(augmented_weights)

for cond_name, cond_fn in [("night", simulate_night), ("blur", simulate_blur)]:
    fig, axes = plt.subplots(2, len(samples), figsize=(7 * len(samples), 10))
    fig.suptitle(f"Condition: {cond_name.upper()} — Top: Baseline | Bottom: Augmented", 
                 fontsize=14, fontweight="bold")
    
    if len(samples) == 1:
        axes = np.array([[axes[0]], [axes[1]]])
        
    for col, img_path in enumerate(samples):
        img = cv2.imread(str(img_path))
        degraded = cond_fn(img.copy())
        
        temp_path = str(DEGRADED_DIR / f"temp_{cond_name}_{col}.jpg")
        cv2.imwrite(temp_path, degraded)
        
        # Baseline prediction
        r_base = baseline_model.predict(temp_path, conf=0.25, device=DEVICE, verbose=False)[0]
        axes[0, col].imshow(cv2.cvtColor(r_base.plot(), cv2.COLOR_BGR2RGB))
        axes[0, col].set_title(f"Baseline: {len(r_base.boxes)} detections", fontsize=10)
        axes[0, col].axis("off")
        
        # Augmented prediction
        r_aug = aug_model.predict(temp_path, conf=0.25, device=DEVICE, verbose=False)[0]
        axes[1, col].imshow(cv2.cvtColor(r_aug.plot(), cv2.COLOR_BGR2RGB))
        axes[1, col].set_title(f"Augmented: {len(r_aug.boxes)} detections", fontsize=10)
        axes[1, col].axis("off")
    
    plt.tight_layout()
    plt.show()

## Step 14 — Export & Download Trained Weights for System Deployment
Packages all 4 core detector weights (`twowheeler_best.pt`, `rider_best.pt`, `helmet_best.pt`, `plate_best.pt`) plus `rider_augmented_best.pt` into a single zip file for direct placement into your project's `weights/` folder.

In [ ]:
weight_mappings = {
    "twowheeler_best.pt": BESTS.get("twowheeler_train_val") or (RUNS_DIR / "twowheeler_train_val" / "weights" / "best.pt"),
    "rider_best.pt": BESTS.get("rider_train_val") or (RUNS_DIR / "rider_train_val" / "weights" / "best.pt"),
    "helmet_best.pt": BESTS.get("helmet_no_helmet_train_val") or (RUNS_DIR / "helmet_no_helmet_train_val" / "weights" / "best.pt"),
    "plate_best.pt": BESTS.get("plate_train_val") or (RUNS_DIR / "plate_train_val" / "weights" / "best.pt"),
    "rider_augmented_best.pt": BESTS.get("rider_augmented") or (RUNS_DIR / "rider_augmented" / "weights" / "best.pt"),
}

print("Exporting trained model weights:")
copied_count = 0
for target_name, src_path in weight_mappings.items():
    if src_path and Path(src_path).exists():
        dst_path = EXPORT_DIR / target_name
        shutil.copy(src_path, dst_path)
        print(f" [+] Copied {target_name} ({Path(src_path).stat().st_size / (1024*1024):.1f} MB)")
        copied_count += 1
    else:
        print(f" ⚠️ Missing {target_name} (Expected at: {src_path})")

# Zip exported weights
zip_export_path = BASE_DIR / "all_trained_weights.zip"
if copied_count > 0:
    shutil.make_archive(str(BASE_DIR / "all_trained_weights"), 'zip', str(EXPORT_DIR))
    print(f"\n📦 Successfully created: {zip_export_path}")
    print(f"Size: {zip_export_path.stat().st_size / (1024*1024):.1f} MB")
    
    if IS_COLAB:
        try:
            from google.colab import files
            print("Initiating automatic download to your browser...")
            files.download(str(zip_export_path))
        except Exception as e:
            print(f"In Colab, download manually from file explorer: {zip_export_path}")
            
    print("\n👉 Deployment Instructions:")
    print("Unzip this archive into your local project's 'weights/' folder:")
    print("  Automatic-Helmet-Detection/weights/")
    print("The cascade pipeline and dashboard will automatically detect and use these weights!")